# Phase II - Step 5: JEV decision layer (typed, calibrated, abstaining)

Turns the per-alert signals from notebook 03 into a **typed decision** for every alert:

| decision | meaning |
|---|---|
| `BLOCK` | calibrated P(attack) >= `t_hi`: act automatically |
| `ALLOW` | calibrated P(attack) < `t_lo`: let through automatically |
| `ESCALATE` | in between: abstain, send to a human analyst (Step 6) |

**Surrogate JEV** = a small calibrated model over `[logit p_RF, entropy, tree_std, flip_rate, ESS, SI, SI_entropy, AR]`.
Thresholds are chosen on the **validation** sample by minimising expected cost per alert
`C_FN` (missed attack) / `C_FP` (false block) / `C_ESC` (analyst time per escalated alert, assumed to resolve it correctly),
using 5-fold out-of-fold predictions so that the thresholds are not tuned on in-sample probabilities.
Everything is then evaluated once on the **test** sample (weighted to the true traffic mix with `sample_weight`).

Comparisons: RF alone at 0.5, RF max-prob abstention (calibrated RF probability only), XAI-Q-only gating (ESS/SI/AR, no probability), JEV full, JEV ablations (no ESS / no SI / no AR / no tree-disagreement), and a GBM variant of the JEV.

Cost values are assumptions, not measurements: the sensitivity table in section 7 shows how conclusions change with them.

## 0. Setup

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
SEED = 42
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

# Static path: WSL Ubuntu view of the D: drive (falls back to the Windows path outside WSL)
ROOT = Path('/mnt/d/ai_security/xai-q-audit-ids') if Path('/mnt/d').exists() else Path(r'D:\ai_security\xai-q-audit-ids')
OUT_DIR = ROOT / 'data' / 'phaseII_outputs'

# Cost model (per alert, relative units). Assumption: a missed attack costs 20 false blocks; a human review costs 0.3.
C_FN, C_FP, C_ESC = 20.0, 1.0, 0.3
N_BOOT = 1000
print('out dir:', OUT_DIR, '| exists:', OUT_DIR.is_dir())

out dir: /mnt/d/ai_security/xai-q-audit-ids/data/phaseII_outputs | exists: True


## 1. Load signals
`val` is used to fit the JEV and choose thresholds; `test` is only used for evaluation.

In [ ]:
val = pd.read_parquet(OUT_DIR / 'xaiq_val.parquet')
test = pd.read_parquet(OUT_DIR / 'xaiq_test.parquet')
print('val', val.shape, '| test', test.shape)
for n, s in [('val', val), ('test', test)]:
    w = s.sample_weight
    print(f'{n}: est. alerts {w.sum():,.0f} | attack share {np.average(s.label, weights=w):.3f} | '
          f'RF error rate @0.5 {np.average(1 - s.correct, weights=w):.3f}')


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def design(df):
    return pd.DataFrame({'logit_p': logit(df.p_attack.values), 'entropy': df.entropy.values, 'tree_std': df.tree_std.values,
                         'flip_rate': df.flip_rate.values, 'ESS': df.ESS.values, 'SI': df.SI.values,
                         'SI_entropy': df.SI_entropy.values, 'AR': df.AR.values}, index=df.index)


Xv, Xt = design(val), design(test)
yv, yt = val.label.values, test.label.values
wv, wt = val.sample_weight.values, test.sample_weight.values
FULL = list(Xv.columns)

## 2. Variants and out-of-fold calibration
Every variant is a logistic regression (or GBM) fitted on the validation sample with sample weights, so its output is a calibrated P(attack) for the *same* validation distribution. Out-of-fold (OOF) probabilities on validation are used to pick thresholds; the model refit on all of validation is applied to test.

In [ ]:
variants = {
    'RF max-prob (calibrated)': (['logit_p'], 'lr'),
    'XAI-Q only (no probability)': (['ESS', 'SI', 'SI_entropy', 'AR'], 'lr'),
    'JEV full': (FULL, 'lr'),
    'JEV - ESS': ([c for c in FULL if c != 'ESS'], 'lr'),
    'JEV - SI': ([c for c in FULL if c not in ('SI', 'SI_entropy')], 'lr'),
    'JEV - AR': ([c for c in FULL if c != 'AR'], 'lr'),
    'JEV - tree/flip': ([c for c in FULL if c not in ('tree_std', 'flip_rate')], 'lr'),
    'JEV full (GBM)': (FULL, 'gbm'),
}


def fit_model(cols, kind, X, y, w):
    w = w / w.mean()
    if kind == 'lr':
        m = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
        m.fit(X[cols], y, logisticregression__sample_weight=w)
    else:
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=0.05, random_state=SEED)
        m.fit(X[cols], y, sample_weight=w)
    return m


skf = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xv, val.attack_cat))
P_oof, P_test, MODELS = {}, {}, {}
for name, (cols, kind) in variants.items():
    oof = np.zeros(len(Xv))
    for tr, te in skf:
        oof[te] = fit_model(cols, kind, Xv.iloc[tr], yv[tr], wv[tr]).predict_proba(Xv.iloc[te][cols])[:, 1]
    MODELS[name] = fit_model(cols, kind, Xv, yv, wv)
    P_oof[name] = oof
    P_test[name] = MODELS[name].predict_proba(Xt[cols])[:, 1]
print('fitted', len(variants), 'variants')

coef = pd.Series(MODELS['JEV full'][-1].coef_[0], index=FULL, name='std. coefficient (JEV full)')
coef.round(3).to_frame()

## 3. Threshold policy (cost-based abstention)
For thresholds `t_lo <= t_hi`: `ALLOW` if p < t_lo, `BLOCK` if p >= t_hi, else `ESCALATE`. The pair minimising weighted expected cost on the OOF validation probabilities is selected (exact search over a 0.0025-spaced grid using cumulative sums).

In [ ]:
GRID = np.r_[np.linspace(0, 1, 401), 1.0001]


def choose_thresholds(p, y, w, c_fn, c_fp, c_esc):
    o = np.argsort(p)
    ps, ws, ys = p[o], w[o], y[o]
    cum_fn = np.r_[0, np.cumsum(ws * ys * c_fn)]
    cum_fp = np.r_[0, np.cumsum(ws * (1 - ys) * c_fp)]
    cum_w = np.r_[0, np.cumsum(ws)]
    i = np.searchsorted(ps, GRID, 'left')
    cost = (cum_fn[i][:, None] + (cum_fp[-1] - cum_fp[i])[None, :] + c_esc * (cum_w[i][None, :] - cum_w[i][:, None]))
    cost = np.where(GRID[None, :] >= GRID[:, None], cost, np.inf) / cum_w[-1]
    a, b = np.unravel_index(np.argmin(cost), cost.shape)
    return GRID[a], GRID[b], cost[a, b]


def typed(p, lo, hi):
    return np.where(p < lo, 'ALLOW', np.where(p >= hi, 'BLOCK', 'ESCALATE'))


def row_cost(d, y, c_fn=C_FN, c_fp=C_FP, c_esc=C_ESC):
    return np.where(d == 'ALLOW', y * c_fn, np.where(d == 'BLOCK', (1 - y) * c_fp, c_esc))


def summarize(d, y, w, **c):
    W = w.sum()
    auto = d != 'ESCALATE'
    esc = ~auto
    fn = (d == 'ALLOW') & (y == 1)
    fp = (d == 'BLOCK') & (y == 0)
    return {
        'cost/alert': np.average(row_cost(d, y, **c), weights=w),
        'escalate %': 100 * w[esc].sum() / W,
        'auto-block %': 100 * w[d == 'BLOCK'].sum() / W,
        'auto-allow %': 100 * w[d == 'ALLOW'].sum() / W,
        'auto error %': 100 * w[fn | fp].sum() / max(w[auto].sum(), 1e-9),
        'missed attacks %': 100 * w[fn].sum() / w[y == 1].sum(),
        'false blocks % of normal': 100 * w[fp].sum() / w[y == 0].sum(),
        'attack share in queue %': 100 * (w[esc & (y == 1)].sum() / max(w[esc].sum(), 1e-9)),
    }


POLICY = {}
for name in variants:
    lo, hi, c = choose_thresholds(P_oof[name], yv, wv, C_FN, C_FP, C_ESC)
    POLICY[name] = (lo, hi)
    print(f'{name:<30s} t_lo={lo:.4f} t_hi={hi:.4f}  val OOF cost={c:.3f}')

## 4. Test results (main cost setting)
`RF alone @0.5` is the paper's model with no abstention (two decisions: block / allow). Lower `cost/alert` is better.

In [ ]:
rows, D_test, RC = {}, {}, {}
d0 = np.where(test.p_attack.values >= 0.5, 'BLOCK', 'ALLOW')
rows['RF alone @0.5 (no abstention)'] = summarize(d0, yt, wt)
D_test['RF alone @0.5 (no abstention)'] = d0
for name in variants:
    d = typed(P_test[name], *POLICY[name])
    D_test[name] = d
    rows[name] = summarize(d, yt, wt)
res = pd.DataFrame(rows).T
res['cost reduction vs RF@0.5 %'] = 100 * (1 - res['cost/alert'] / res.loc['RF alone @0.5 (no abstention)', 'cost/alert'])
res.round(3)

## 5. Uncertainty around the comparison (stratified bootstrap of the test sample)
Test alerts are resampled with replacement inside each attack category; thresholds and models stay fixed. Differences are paired (same resample for both methods).

In [ ]:
rng = np.random.default_rng(SEED)
cats = test.attack_cat.values
groups = [np.where(cats == c)[0] for c in np.unique(cats)]
idx = np.concatenate([g[rng.integers(0, len(g), (N_BOOT, len(g)))] for g in groups], axis=1)
RC = {n: row_cost(d, yt) for n, d in D_test.items()}


def boot(c):
    return (wt[idx] * c[idx]).sum(1) / wt[idx].sum(1)


B = {n: boot(c) for n, c in RC.items()}
ref = 'RF max-prob (calibrated)'
tab = []
for n in B:
    lo, mid, hi = np.percentile(B[n], [2.5, 50, 97.5])
    diff = B[n] - B[ref]
    dlo, dhi = np.percentile(diff, [2.5, 97.5])
    tab.append({'method': n, 'cost/alert': RC[n] @ wt / wt.sum(), 'CI low': lo, 'CI high': hi,
                f'diff vs {ref}': diff.mean(), 'diff CI low': dlo, 'diff CI high': dhi, 'P(better)': (diff < 0).mean()})
pd.DataFrame(tab).set_index('method').round(3)

## 6. Calibration and selective risk
- **Calibration:** weighted ECE (10 bins) and Brier score of the RF's raw probability, the calibrated RF probability and the JEV probability.
- **Selective risk:** error rate of the alerts kept when the most uncertain alerts are dropped first (lower area = better ranking of which alerts to trust). Uncertainty = `1 - max(p, 1-p)`.

In [ ]:
def ece(p, y, w, bins=10):
    b = np.minimum((p * bins).astype(int), bins - 1)
    return sum(w[b == k].sum() / w.sum() * abs(np.average(y[b == k], weights=w[b == k]) - np.average(p[b == k], weights=w[b == k]))
               for k in range(bins) if (b == k).any())


def brier(p, y, w):
    return np.average((p - y) ** 2, weights=w)


probs = {'RF raw': test.p_attack.values, 'RF calibrated': P_test['RF max-prob (calibrated)'], 'XAI-Q only': P_test['XAI-Q only (no probability)'],
         'JEV full': P_test['JEV full'], 'JEV full (GBM)': P_test['JEV full (GBM)']}
cal = pd.DataFrame({k: {'ECE': ece(p, yt, wt), 'Brier': brier(p, yt, wt), 'AUROC': roc_auc_score(yt, p, sample_weight=wt)}
                    for k, p in probs.items()}).T
cal.round(4)

In [ ]:
def risk_coverage(p, y, w):
    u = 1 - np.maximum(p, 1 - p)
    err = ((p >= 0.5).astype(int) != y).astype(float)
    o = np.argsort(u, kind='stable')
    cw = np.cumsum(w[o])
    cov = cw / cw[-1]
    risk = np.cumsum(w[o] * err[o]) / cw
    return cov, risk


curves = {k: risk_coverage(p, yt, wt) for k, p in probs.items()}
aurc = pd.Series({k: np.trapz(r, c) for k, (c, r) in curves.items()}, name='AURC (lower is better)')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, (c, r) in curves.items():
    ax[0].plot(c, r, label=k)
ax[0].set_xlabel('coverage (fraction auto-decided)')
ax[0].set_ylabel('selective risk (error rate)')
ax[0].legend()
ax[0].set_title('Risk-coverage (test, weighted)')
for k in ['RF raw', 'JEV full']:
    p = probs[k]
    bins = np.minimum((p * 10).astype(int), 9)
    xs = [np.average(p[bins == j], weights=wt[bins == j]) for j in range(10) if (bins == j).any()]
    ys = [np.average(yt[bins == j], weights=wt[bins == j]) for j in range(10) if (bins == j).any()]
    ax[1].plot(xs, ys, marker='o', label=k)
ax[1].plot([0, 1], [0, 1], 'k--', lw=0.8)
ax[1].set_xlabel('predicted P(attack)')
ax[1].set_ylabel('observed attack rate')
ax[1].legend()
ax[1].set_title('Reliability (test, weighted)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'jev_calibration_risk_coverage.png', dpi=150)
plt.show()
aurc.round(4).to_frame()

## 7. Sensitivity to the cost assumptions
Thresholds are re-selected on validation for each cost setting; the table reports test cost per alert and the reduction relative to RF alone at 0.5 under the *same* costs.

In [ ]:
settings = [(10, 1, 0.3), (20, 1, 0.3), (50, 1, 0.3), (20, 1, 0.1), (20, 1, 0.6), (20, 1, 1.0)]
focus = ['RF max-prob (calibrated)', 'XAI-Q only (no probability)', 'JEV full', 'JEV full (GBM)']
srows = []
for c_fn, c_fp, c_esc in settings:
    base = np.average(row_cost(d0, yt, c_fn, c_fp, c_esc), weights=wt)
    r = {'C_FN': c_fn, 'C_ESC': c_esc, 'RF@0.5 cost': base}
    for n in focus:
        lo, hi, _ = choose_thresholds(P_oof[n], yv, wv, c_fn, c_fp, c_esc)
        d = typed(P_test[n], lo, hi)
        cst = np.average(row_cost(d, yt, c_fn, c_fp, c_esc), weights=wt)
        r[f'{n} reduction %'] = 100 * (1 - cst / base)
        r[f'{n} esc %'] = 100 * wt[d == 'ESCALATE'].sum() / wt.sum()
    srows.append(r)
pd.DataFrame(srows).round(2)

## 8. Where does JEV act, per attack category (test)

In [ ]:
d = D_test['JEV full']
rows = {}
for c in np.unique(cats):
    m = cats == c
    s = summarize(d[m], yt[m], wt[m])
    rows[c] = {'est. alerts': wt[m].sum(), 'block %': s['auto-block %'], 'allow %': s['auto-allow %'], 'escalate %': s['escalate %'],
               'auto error %': s['auto error %'], 'RF error % @0.5': 100 * np.average(1 - test.correct.values[m], weights=wt[m])}
pd.DataFrame(rows).T.round(2)

## 9. Save for the human-triage step

In [ ]:
out = test[['attack_cat', 'label', 'sample_weight', 'p_attack', 'ESS', 'SI', 'AR', 'tree_std', 'top_feats']].copy()
out['p_jev'] = P_test['JEV full']
out['decision'] = D_test['JEV full']
out['decision_rf_maxprob'] = D_test['RF max-prob (calibrated)']
out.to_parquet(OUT_DIR / 'jev_test_decisions.parquet')
oofdf = val[['attack_cat', 'label', 'sample_weight']].copy()
oofdf['p_jev_oof'] = P_oof['JEV full']
oofdf.to_parquet(OUT_DIR / 'jev_val_oof.parquet')
config = {'costs': {'C_FN': C_FN, 'C_FP': C_FP, 'C_ESC': C_ESC}, 'features': FULL,
          'thresholds': {k: {'t_lo': float(v[0]), 't_hi': float(v[1])} for k, v in POLICY.items()},
          'coef_jev_full': {k: float(v) for k, v in coef.items()}}
json.dump(config, open(OUT_DIR / 'jev_config.json', 'w'), indent=1)
res.round(4).to_csv(OUT_DIR / 'jev_results_test.csv')
print(out.decision.value_counts())